## merged_basic_descriptors.csv を作成

このノートブックでは、以下の順にCSVを結合して `merged_basic_descriptors.csv` を作成します。

1. `structures_basic_descriptors.csv` と `smiles_list_merged.csv` を `smiles` で結合  
2. `composition.csv` を `material_id` で結合  
3. `wvp_test.csv` を `composition_id` で結合  

最後に、指定の列順に整形してCSV出力します。

In [1]:
from pathlib import Path
import pandas as pd

base = Path("../../data")
paths = {
    "desc": base / "interim" / "structures_basic_descriptors.csv",
    "smiles": base / "interim" / "smiles_list_merged.csv",
    "comp": base / "raw" / "composition.csv",
    "wvp": base / "raw" / "wvp_test.csv",
    "out": base / "interim" / "merged_basic_descriptors.csv",
}

## 1. データ読み込み

- 記述子: `structures_all_descriptors.csv`
- SMILES一覧: `smiles_list_merged.csv`
- 組成: `composition.csv`
- 実験結果: `wvp_test.csv`

※ `structures_all_descriptors.csv` の `id` は、出力では `structure_id` として扱います。

In [2]:
desc = pd.read_csv(paths["desc"]).rename(columns={"id": "structure_id"})
smiles = pd.read_csv(paths["smiles"])[["material_id", "name", "smiles", "ratio"]]
comp = pd.read_csv(paths["comp"])[["composition_id", "material_id", "concentration"]].rename(
    columns={"concentration": "material_concentration"}
)
wvp = pd.read_csv(paths["wvp"])

## 2. 最低限の前処理（結合キーの整形）

- `smiles` は前後の空白を除去
- `material_id`, `composition_id` などは数値として扱えるように変換

※ `wvp_test.csv` はヘッダーに `id` が重複していますが、ここでは使用しません（`experiment_id`, `composition_id`, `temperature`, `humidity`, `wvp` のみ利用）。

In [3]:
for df in (desc, smiles):
    df["smiles"] = df["smiles"].astype(str).str.strip()

smiles["material_id"] = pd.to_numeric(smiles["material_id"], errors="coerce").astype("Int64")
comp["material_id"] = pd.to_numeric(comp["material_id"], errors="coerce").astype("Int64")
comp["composition_id"] = pd.to_numeric(comp["composition_id"], errors="coerce").astype("Int64")

wvp["experiment_id"] = pd.to_numeric(wvp["experiment_id"], errors="coerce").astype("Int64")
wvp["composition_id"] = pd.to_numeric(wvp["composition_id"], errors="coerce").astype("Int64")

## 3. 結合

指定どおりのキーで順番に結合します。

- 記述子 × SMILES（`smiles`）
- その結果 × composition（`material_id`）
- その結果 × wvp（`composition_id`）

また、`proportion` は `ratio` と同じ意味なので、列名を `ratio → proportion` に変更します。

In [4]:
df = desc.merge(smiles, on="smiles", how="left")

available_material_ids = set(df["material_id"].dropna().unique())
valid_composition_ids = (
    comp.groupby("composition_id")["material_id"]
    .apply(lambda s: s.isin(available_material_ids).all())
)
comp = comp[comp["composition_id"].isin(valid_composition_ids[valid_composition_ids].index)]

df = df.merge(comp, on="material_id", how="inner")

wvp_small = wvp[["experiment_id", "composition_id", "temperature", "humidity", "wvp"]]
df = df.merge(wvp_small, on="composition_id", how="left")

df = df.rename(columns={"ratio": "proportion"})

## 4. 列順の整形とCSV出力

先頭に以下の列を並べ、その後に「記述子列」をまとめて入れ、最後に環境条件と目的変数（`humidity`, `temperature`, `wvp`）を置きます。

出力列（先頭）:
- `experiment_id`
- `composition_id`
- `material_id`
- `name`
- `structure_id`
- `material_concentration`
- `smiles`
- `proportion`

最後:
- `humidity`
- `temperature`
- `wvp`

In [5]:
base_cols = [
    "experiment_id", "composition_id", "material_id", "name",
    "structure_id", "material_concentration", "smiles", "proportion",
]
env_cols = ["humidity", "temperature", "wvp"]
desc_cols = [c for c in df.columns if c not in set(base_cols + env_cols)]

# 列順を整形
df = df[base_cols + desc_cols + env_cols]

# 列順を整形
df = df[base_cols + desc_cols + env_cols]

# wvp が欠損 or 0.0 を削除（念のため数値化してから判定）
df["wvp"] = pd.to_numeric(df["wvp"], errors="coerce")
df = df[df["wvp"].notna() & (df["wvp"] != 0.0)].copy()

# まったく同じ行を削除（全列一致）
df = df.drop_duplicates(keep="first").reset_index(drop=True)

# composition_id順に並べて、idを振る
df = df.sort_values(["composition_id"], kind="stable", na_position="last").reset_index(drop=True)


## 5．各組成に水を追加

In [6]:
# composition_idごとの最後に water 行を追加（参照: smiles=='O'）
w_desc = desc[desc["smiles"].eq("O")]
w_smiles = smiles[smiles["smiles"].eq("O")]
if len(w_desc) != 1 or len(w_smiles) != 1:
    raise ValueError(f"smiles=='O' must be unique: desc={len(w_desc)}, smiles={len(w_smiles)}")

w_desc = w_desc.iloc[0]
w_smiles = w_smiles.iloc[0]
w_desc_cols = [c for c in desc.columns if c not in {"structure_id", "smiles"}]

chunks = []
for _, g in df.groupby("composition_id", sort=False):
    water = g.iloc[-1].copy()
    water["material_id"] = 144
    water["material_concentration"] = 100
    water["name"] = w_smiles["name"]
    water["structure_id"] = w_desc["structure_id"]
    water["smiles"] = "O"
    water["proportion"] = w_smiles["ratio"]
    for c in w_desc_cols:
        if c in water.index:
            water[c] = w_desc[c]
    chunks.append(pd.concat([g, water.to_frame().T], ignore_index=True))

df = pd.concat(chunks, ignore_index=True)

## 6．idを振ってcsv保存

In [7]:
df.insert(0, "id", range(1, len(df) + 1))

df.to_csv(paths["out"], index=False, encoding="utf-8-sig")
df.head()

,id,experiment_id,composition_id,material_id,name,structure_id,material_concentration,smiles,proportion,ExactMolWt,...,NumRotatableBonds,FractionCSP3,LabuteASA,BalabanJ,HeavyAtomCount,NOCount,NumAromaticRings,humidity,temperature,wvp
0,1,18,64,3,cellulose nanofiber,2,0.5,OC[C@H]1O[C@@H](O)[C@H](O)[C@@H](O)[C@@H]1O,100.0,180.063388,...,1.0,1.0,68.642844,2.634409,12.0,6.0,0.0,90.0,25.0,1.35
1,2,18,64,77,carboxy-methyl cellulose,23,1.0,C([C@@H]1[C@H]([C@@H]([C@H]([C@@H](O1)O)O)O)O)O,100.0,180.063388,...,1.0,1.0,68.642844,2.634409,12.0,6.0,0.0,90.0,25.0,1.35
2,3,18,64,23,glycerol,45,0.35,OCC(O)CO,100.0,92.047344,...,2.0,1.0,35.851837,2.754185,6.0,3.0,0.0,90.0,25.0,1.35
3,4,18,64,144,water,119,100,O,100.0,18.010565,...,0.0,0.0,6.849231,0.0,1.0,1.0,0.0,90.0,25.0,1.35
4,5,18,65,3,cellulose nanofiber,2,0.5,OC[C@H]1O[C@@H](O)[C@H](O)[C@@H](O)[C@@H]1O,100.0,180.063388,...,1.0,1.0,68.642844,2.634409,12.0,6.0,0.0,90.0,25.0,1.36
